# 🛒 Project 004 — E-commerce Churn Prediction
**Portfolio project | Ishan Sewnandan**

Identifying at-risk customers for a Dutch retail case using XGBoost classification.  
Tools: Python · Pandas · XGBoost · Scikit-learn · Excel

---
### Doel
- Klanten met hoog churnrisico vroegtijdig identificeren
- XGBoost classificatiemodel trainen op gedragskenmerken
- Business aanbevelingen formuleren per risicocategorie
- Resultaten exporteren naar Excel voor stakeholder rapportage

### Dataset
- **Kaggle: E-commerce Customer Churn** (5.630 klanten, 20 features)
- Gecontextualiseerd als Nederlandse retail case
- Churn definitie: geen aankoop in afgelopen 30+ dagen (inactief klant)


## 0. Setup & Data Ophalen

In [ ]:
# !pip install xgboost scikit-learn pandas numpy plotly openpyxl shap kaggle

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')

print('✅ Alle packages geladen')
print(f'   XGBoost versie: {xgb.__version__}')

In [ ]:
# ─── Data ophalen via Kaggle API ──────────────────────────────────────────────
# Vereist: kaggle.json in ~/.kaggle/ (gratis account op kaggle.com)
# Zie: https://www.kaggle.com/docs/api

import os

DATA_FILE = 'E Commerce.xlsx'

if not os.path.exists(DATA_FILE):
    print('📥 Data downloaden via Kaggle API...')
    os.system('kaggle datasets download -d ankitverma2010/ecommerce-customer-churn-analysis-and-prediction --unzip')
    print('✅ Download klaar')
else:
    print(f'✅ Data al aanwezig: {DATA_FILE}')

# Alternatief: handmatig downloaden van:
# https://www.kaggle.com/datasets/ankitverma2010/ecommerce-customer-churn-analysis-and-prediction
# Zet het Excel-bestand in dezelfde map als deze notebook

In [ ]:
# ─── Data inladen ─────────────────────────────────────────────────────────────
df = pd.read_excel(DATA_FILE, sheet_name='E Comm')
print(f'Dataset: {df.shape[0]:,} rijen × {df.shape[1]} kolommen')
print(f'Churn rate: {df["Churn"].mean()*100:.1f}%')
df.head()

## 1. Exploratory Data Analysis

In [ ]:
# ─── Dataset overzicht ────────────────────────────────────────────────────────
print('Kolominfo:')
print(df.dtypes)
print(f'\nMissing values per kolom:')
missing = df.isnull().sum()
print(missing[missing > 0])

In [ ]:
# ─── Churn distributie ────────────────────────────────────────────────────────
churn_counts = df['Churn'].value_counts()
print(f'Actief:   {churn_counts[0]:,} klanten ({churn_counts[0]/len(df)*100:.1f}%)')
print(f'Churned:  {churn_counts[1]:,} klanten ({churn_counts[1]/len(df)*100:.1f}%)')

fig1 = px.pie(
    values=churn_counts.values,
    names=['Actief', 'Churned'],
    title='🎯 Churn Distributie',
    color_discrete_sequence=['#5C8A3C', '#EF4444'],
    template='plotly_white'
)
fig1.update_layout(font_family='Georgia')
fig1.show()

In [ ]:
# ─── Churn rate per categorie ─────────────────────────────────────────────────
cat_cols = ['PreferredLoginDevice', 'PreferredPaymentMode', 
            'Gender', 'PreferedOrderCat', 'MaritalStatus']

# Filter op bestaande kolommen
cat_cols = [c for c in cat_cols if c in df.columns]

fig2 = make_subplots(
    rows=2, cols=3,
    subplot_titles=cat_cols
)

for i, col in enumerate(cat_cols):
    r, c = divmod(i, 3)
    churn_by = df.groupby(col)['Churn'].mean().reset_index()
    churn_by.columns = [col, 'churn_rate']
    churn_by = churn_by.sort_values('churn_rate', ascending=False)
    
    fig2.add_trace(
        go.Bar(x=churn_by[col], y=churn_by['churn_rate'],
               marker_color=['#EF4444' if v > df['Churn'].mean() else '#5C8A3C' 
                             for v in churn_by['churn_rate']],
               showlegend=False),
        row=r+1, col=c+1
    )

fig2.update_layout(
    title='📊 Churn Rate per Categorische Feature',
    template='plotly_white', font_family='Georgia',
    height=500
)
fig2.update_yaxes(tickformat='.0%')
fig2.show()

In [ ]:
# ─── Numerieke features vs churn ─────────────────────────────────────────────
num_cols = ['Tenure', 'WarehouseToHome', 'HourSpendOnApp', 
            'NumberOfDeviceRegistered', 'SatisfactionScore',
            'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear',
            'CouponUsed', 'OrderCount', 'DaySinceLastOrder']
num_cols = [c for c in num_cols if c in df.columns]

stats = df.groupby('Churn')[num_cols].mean().T
stats.columns = ['Actief (0)', 'Churned (1)']
stats['Verschil %'] = ((stats['Churned (1)'] - stats['Actief (0)']) / stats['Actief (0)'] * 100).round(1)
stats = stats.sort_values('Verschil %', key=abs, ascending=False)

print('📈 Gemiddelde waarden: Actief vs Churned')
stats.style.format({'Actief (0)': '{:.2f}', 'Churned (1)': '{:.2f}', 'Verschil %': '{:+.1f}%'})\
     .background_gradient(subset=['Verschil %'], cmap='RdYlGn_r')

## 2. Data Preprocessing

In [ ]:
# ─── Kolom CustomerID verwijderen ─────────────────────────────────────────────
df_model = df.copy()
if 'CustomerID' in df_model.columns:
    df_model = df_model.drop(columns=['CustomerID'])

# ─── Missing values behandelen ───────────────────────────────────────────────
# Numeriek: mediaan per churn-groep imputen
num_cols_all = df_model.select_dtypes(include=[np.number]).columns.tolist()
num_cols_all = [c for c in num_cols_all if c != 'Churn']

for col in num_cols_all:
    if df_model[col].isnull().sum() > 0:
        median_by_churn = df_model.groupby('Churn')[col].transform('median')
        df_model[col] = df_model[col].fillna(median_by_churn)

# Categorisch: modus
cat_cols_all = df_model.select_dtypes(include='object').columns.tolist()
for col in cat_cols_all:
    df_model[col] = df_model[col].fillna(df_model[col].mode()[0])

print(f'Missing values na imputing: {df_model.isnull().sum().sum()}')

In [ ]:
# ─── Feature Engineering ─────────────────────────────────────────────────────
# Recency-Frequency-Monetary (RFM) proxies
if 'DaySinceLastOrder' in df_model.columns:
    df_model['recency_segment'] = pd.cut(
        df_model['DaySinceLastOrder'],
        bins=[0, 7, 14, 30, 100],
        labels=['<1w', '1-2w', '2-4w', '>4w']
    ).astype(str)

if 'OrderCount' in df_model.columns and 'Tenure' in df_model.columns:
    df_model['orders_per_month'] = df_model['OrderCount'] / (df_model['Tenure'] + 1)

if 'SatisfactionScore' in df_model.columns and 'Complain' in df_model.columns:
    df_model['risk_score'] = df_model['Complain'] * (6 - df_model['SatisfactionScore'])

if 'CouponUsed' in df_model.columns and 'OrderCount' in df_model.columns:
    df_model['coupon_dependency'] = df_model['CouponUsed'] / (df_model['OrderCount'] + 1)

print('✅ Feature engineering klaar')
print(f'Totaal features: {df_model.shape[1] - 1}')

In [ ]:
# ─── Label encoding categorische features ────────────────────────────────────
cat_cols_final = df_model.select_dtypes(include='object').columns.tolist()
le_dict = {}

for col in cat_cols_final:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    le_dict[col] = le

print(f'Label-encoded: {cat_cols_final}')

# ─── Train/test split (stratified) ───────────────────────────────────────────
X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTrain: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Churn rate train: {y_train.mean()*100:.1f}% | test: {y_test.mean()*100:.1f}%')

## 3. Model Training & Vergelijking

In [ ]:
# ─── Drie modellen trainen ────────────────────────────────────────────────────
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()  # class imbalance correctie

modellen = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, C=0.1),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=8, random_state=42, class_weight='balanced'
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        scale_pos_weight=scale_pos,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, eval_metric='logloss',
        use_label_encoder=False
    )
}

resultaten = {}

for naam, model in modellen.items():
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    
    auc    = roc_auc_score(y_test, y_prob)
    ap     = average_precision_score(y_test, y_prob)
    report = classification_report(y_test, y_pred, output_dict=True)
    
    resultaten[naam] = {
        'model': model, 'y_prob': y_prob, 'y_pred': y_pred,
        'AUC-ROC': auc, 'Avg Precision': ap,
        'Precision (churn)': report['1']['precision'],
        'Recall (churn)': report['1']['recall'],
        'F1 (churn)': report['1']['f1-score']
    }
    
    print(f'\n{naam}')
    print(f'  AUC-ROC:    {auc:.4f}')
    print(f'  Precision:  {report["1"]["precision"]:.4f}')
    print(f'  Recall:     {report["1"]["recall"]:.4f}')
    print(f'  F1-score:   {report["1"]["f1-score"]:.4f}')

In [ ]:
# ─── ROC curve vergelijking ───────────────────────────────────────────────────
fig3 = go.Figure()
colors = {'Logistic Regression': '#888', 'Random Forest': '#3B82F6', 'XGBoost': '#5C8A3C'}

for naam, res in resultaten.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    fig3.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines',
        name=f"{naam} (AUC={res['AUC-ROC']:.3f})",
        line=dict(color=colors.get(naam, '#333'), width=2.5)
    ))

fig3.add_trace(go.Scatter(
    x=[0,1], y=[0,1], mode='lines',
    line=dict(color='gray', dash='dash', width=1),
    name='Random baseline', showlegend=True
))

fig3.update_layout(
    title='📈 ROC Curves — Model Vergelijking',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    template='plotly_white',
    font_family='Georgia'
)
fig3.show()

In [ ]:
# ─── Beste model: XGBoost ─────────────────────────────────────────────────────
beste_naam = max(resultaten, key=lambda k: resultaten[k]['AUC-ROC'])
beste_model = resultaten[beste_naam]['model']
print(f'🏆 Beste model: {beste_naam}')
print(f'   AUC-ROC: {resultaten[beste_naam]["AUC-ROC"]:.4f}')
print(f'   F1 (churn): {resultaten[beste_naam]["F1 (churn)"]:.4f}')

## 4. Model Interpretatie — SHAP

In [ ]:
# ─── SHAP feature importance ──────────────────────────────────────────────────
xgb_model = resultaten['XGBoost']['model']
explainer  = shap.TreeExplainer(xgb_model)
shap_vals  = explainer.shap_values(X_test)

# Top features
shap_df = pd.DataFrame({
    'feature': X_test.columns,
    'mean_abs_shap': np.abs(shap_vals).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False).head(15)

fig4 = px.bar(
    shap_df.sort_values('mean_abs_shap'),
    x='mean_abs_shap', y='feature',
    orientation='h',
    title='🔍 SHAP Feature Importance — XGBoost Churn Model',
    color='mean_abs_shap',
    color_continuous_scale='Greens',
    template='plotly_white',
    labels={'mean_abs_shap': 'Mean |SHAP|', 'feature': 'Feature'}
)
fig4.update_layout(font_family='Georgia', showlegend=False)
fig4.show()
fig4.write_html('churn_shap.html')
print('✅ Opgeslagen als churn_shap.html')

## 5. Churnrisico Segmentatie

In [ ]:
# ─── Risicoscore aan alle klanten toewijzen ───────────────────────────────────
df_scores = df.copy()
if 'CustomerID' not in df_scores.columns:
    df_scores['CustomerID'] = range(1, len(df_scores)+1)

# Hergebruik preprocessing
df_pred = df_model.drop(columns=['Churn'])
churn_probs = beste_model.predict_proba(df_pred)[:, 1]

df_scores['churn_prob']    = churn_probs
df_scores['churn_pred']    = (churn_probs >= 0.5).astype(int)
df_scores['risicosegment'] = pd.cut(
    churn_probs,
    bins=[0, 0.3, 0.6, 0.8, 1.0],
    labels=['🟢 Laag (<30%)', '🟡 Matig (30-60%)', '🟠 Hoog (60-80%)', '🔴 Kritiek (>80%)']
)

print('Verdeling risicosegmenten:')
seg_counts = df_scores['risicosegment'].value_counts().sort_index()
for seg, count in seg_counts.items():
    print(f'  {seg}: {count:,} klanten ({count/len(df_scores)*100:.1f}%)')

In [ ]:
# ─── Plot: Churnkans distributie ──────────────────────────────────────────────
fig5 = px.histogram(
    df_scores, x='churn_prob',
    color='risicosegment',
    nbins=40,
    title='🎯 Distributie Churnkans — Alle Klanten',
    labels={'churn_prob': 'Voorspelde Churnkans', 'count': 'Aantal Klanten'},
    color_discrete_sequence=['#5C8A3C', '#F59E0B', '#FB923C', '#EF4444'],
    template='plotly_white'
)
fig5.update_layout(font_family='Georgia', bargap=0.05)
fig5.add_vline(x=0.5, line_dash='dash', line_color='gray',
                annotation_text='Drempelwaarde 50%')
fig5.show()
fig5.write_html('churn_distribution.html')
print('✅ Opgeslagen als churn_distribution.html')

## 6. Business Aanbevelingen per Segment

In [ ]:
# ─── Profiel per risicosegment ────────────────────────────────────────────────
profile_cols = [c for c in ['Tenure', 'SatisfactionScore', 'Complain', 
                              'OrderCount', 'DaySinceLastOrder', 'CouponUsed'] 
                if c in df_scores.columns]

segment_profiles = df_scores.groupby('risicosegment')[profile_cols].mean().round(2)
print('Klantprofiel per risicosegment:')
segment_profiles

In [ ]:
# ─── Business aanbevelingen ───────────────────────────────────────────────────
aanbevelingen = {
    '🟢 Laag (<30%)': {
        'actie': 'Loyaliteitsprogramma',
        'beschrijving': 'Behoud momentum. Bied exclusieve voordelen voor trouwe klanten. Focus op upsell en cross-sell.',
        'kanaal': 'E-mail nieuwsbrief, loyalty app',
        'budget': 'Laag — €1–3 per klant'
    },
    '🟡 Matig (30-60%)': {
        'actie': 'Re-engagement campagne',
        'beschrijving': 'Stuur gepersonaliseerde aanbiedingen op basis van aankoophistorie. Kortingsvoucher na inactiviteit.',
        'kanaal': 'Push notificatie, e-mail',
        'budget': 'Gemiddeld — €5–10 per klant'
    },
    '🟠 Hoog (60-80%)': {
        'actie': 'Proactieve retentie',
        'beschrijving': 'Directe outreach door klantenservice. Klachtenafhandeling versnellen. Win-back aanbieding met hoge korting.',
        'kanaal': 'Telefoon, WhatsApp, e-mail',
        'budget': 'Hoog — €15–25 per klant'
    },
    '🔴 Kritiek (>80%)': {
        'actie': 'Last-chance interventie',
        'beschrijving': 'Maximale retentie-inspanning. Persoonlijk contactmoment. Evalueer of klantwaarde (CLV) interventie rechtvaardigt.',
        'kanaal': 'Persoonlijk gesprek, account manager',
        'budget': 'Maximaal — €30–50 per klant (indien CLV > €200)'
    }
}

print('\n💼 BUSINESS AANBEVELINGEN PER RISICOSEGMENT\n')
print('='*60)
for seg, advies in aanbevelingen.items():
    count = (df_scores['risicosegment'] == seg).sum()
    print(f'\n{seg} — {count:,} klanten')
    print(f'  Actie:   {advies["actie"]}')
    print(f'  Aanpak:  {advies["beschrijving"]}')
    print(f'  Kanaal:  {advies["kanaal"]}')
    print(f'  Budget:  {advies["budget"]}')

## 7. Export naar Excel (Stakeholder Rapport)

In [ ]:
# ─── Excel rapportage ─────────────────────────────────────────────────────────
output_file = 'churn_rapport.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    
    # Sheet 1: Alle klanten met risicoscore
    export_cols = ['CustomerID', 'churn_prob', 'churn_pred', 'risicosegment'] + \
                  [c for c in ['Tenure', 'SatisfactionScore', 'Complain',
                               'OrderCount', 'DaySinceLastOrder'] if c in df_scores.columns]
    df_scores[export_cols].sort_values('churn_prob', ascending=False)\
                          .to_excel(writer, sheet_name='Klant Risicoscores', index=False)
    
    # Sheet 2: Segment samenvatting
    seg_summary = df_scores.groupby('risicosegment').agg(
        aantal=('CustomerID', 'count'),
        gem_churnkans=('churn_prob', 'mean'),
        gem_tevredenheid=('SatisfactionScore', 'mean') if 'SatisfactionScore' in df_scores.columns else ('churn_prob', 'count')
    ).round(3)
    seg_summary.to_excel(writer, sheet_name='Segment Samenvatting')
    
    # Sheet 3: Model performance
    perf_data = pd.DataFrame([
        {**{'Model': naam}, 
         **{k: v for k, v in res.items() if k not in ['model', 'y_prob', 'y_pred']}}
        for naam, res in resultaten.items()
    ])
    perf_data.to_excel(writer, sheet_name='Model Performance', index=False)
    
    # Sheet 4: Top 10 SHAP features
    shap_df.to_excel(writer, sheet_name='Feature Importance', index=False)

print(f'✅ Excel rapport opgeslagen als: {output_file}')
print(f'   Sheets: Klant Risicoscores | Segment Samenvatting | Model Performance | Feature Importance')

## 8. Samenvatting & Conclusies

In [ ]:
# ─── Eindoverzicht ────────────────────────────────────────────────────────────
xgb_res = resultaten['XGBoost']
hoog_kritiek = df_scores['risicosegment'].isin(['🟠 Hoog (60-80%)', '🔴 Kritiek (>80%)'])

print('='*60)
print('📋 PROJECT 004 — CONCLUSIES')
print('='*60)
print(f'\n🏆 Beste model: XGBoost')
print(f'   AUC-ROC:   {xgb_res["AUC-ROC"]:.4f}')
print(f'   Precision: {xgb_res["Precision (churn)"]:.4f}')
print(f'   Recall:    {xgb_res["Recall (churn)"]:.4f}')
print(f'   F1-score:  {xgb_res["F1 (churn)"]:.4f}')
print(f'\n🎯 Klanten met hoog/kritiek risico: {hoog_kritiek.sum():,} ({hoog_kritiek.mean()*100:.1f}%)')
print(f'\n✅ Gegenereerde bestanden:')
print(f'   churn_rapport.xlsx      → Excel stakeholder rapport (4 sheets)')
print(f'   churn_shap.html         → SHAP feature importance')
print(f'   churn_distribution.html → Churnkans distributie')
print(f'\n🚀 Klaar voor portfolio!')

---
## 📁 Gegenereerde bestanden

| Bestand | Beschrijving |
|---|---|
| `churn_rapport.xlsx` | Stakeholder rapport — 4 sheets (scores, segmenten, performance, SHAP) |
| `churn_shap.html` | SHAP feature importance visualisatie (Plotly) |
| `churn_distribution.html` | Distributie churnkansen per risicosegment (Plotly) |

## 🔗 Dataset
- [Kaggle: E-commerce Customer Churn](https://www.kaggle.com/datasets/ankitverma2010/ecommerce-customer-churn-analysis-and-prediction)
- 5.630 klanten · 20 features · 16.8% churnrate

## ▶️ Starten
```bash
pip install xgboost shap scikit-learn pandas plotly openpyxl kaggle
# Zorg dat kaggle.json in ~/.kaggle/ staat
jupyter notebook churn_prediction.ipynb
```

---
*Project 004 | Ishan Sewnandan | Rotterdam, 2025*